# Feature engineering at the forecast cutoff

Implements [`.claude/specs/06.1-spec05-features.md`](../../.claude/specs/06.1-spec05-features.md).

[`feature-engineering-magc.ipynb`](feature-engineering-magc.ipynb) (spec 05) builds a leakage-safe
feature set for a forecast origin at **00:00 on `DAY`**. The regression notebook
[`regression-models-claude.ipynb`](../05_modeling/regression-models-claude.ipynb) (spec 06)
forecasts from an earlier origin: the **availability cutoff** on `DAY−1`, i.e. the issue time
(18:00) minus the publication lag of the actuals. Under that origin, five of spec 05's twelve
features read hours of `DAY−1` that are not yet published when the forecast is issued.

This notebook rebuilds spec 05's feature set **for spec 06's origin**: same definitions, with every
look-back moved so that it ends at the cutoff.

## What this notebook produces

- the forecast setting of spec 06, repeated, and an availability check for every look-back
- 16 feature columns in a separate `features` frame, one row per hour of the record
- a **truncation leakage test**: every feature is rebuilt from data cut at its day's cutoff and
  must not change, plus a control that shows the test catches spec 05's 24 h lag
- one export, `data/features/residual_load_features_cutoff.csv`, with each row's `cutoff`

## Not in this notebook

- fitting any model, choosing a split, feature selection
- loading this export into the regression notebook (a later step)
- any feature from `fc_grid_load` or `fc_residual_load`
- changes to spec 05's notebook or its export `data/features/residual_load_features.csv`

---

## 1 Setup and configuration

### 1.1 Shared setup (inherited)

Same setup as [`regression-models-claude.ipynb`](../05_modeling/regression-models-claude.ipynb) §1.1,
which repeats [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1. Inherited, not re-derived:

- the data-directory resolver (walks **upward** from the working directory)
- loading, renaming, the German-CSV float conversion and the dtype asserts
- `time_series`, `SERIES`, `DERIVED`, `YEARS`, `DAY_NAMES`, the season mapping

Not needed here, so not repeated: the model libraries, `style_timeseries` and the plotting helpers.
This notebook draws no time-series plot.

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it with
`notebooks/API-connection.ipynb`.

In [1]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

# Walk up from the working directory to the first parent holding a `data/` folder
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__} · holidays {holidays.__version__}")
print(f"Data directory: {DATA}")

pandas 3.0.5 · numpy 2.5.3 · holidays 0.104
Data directory: /Users/magc/neuefische_course/repositories/grid-stress/data/smard.csv


In [2]:
# The CSV headers from API-connection
# Update this if we include more direct API features!
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")


# Check columns for new entries (new raw features from the API). pipe (|) is a union for a set
assert set(raw.columns) == {"timestamp"} | COLUMNS.keys(), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | COLUMNS.keys()))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(
    raw["timestamp"],
    format="%Y-%m-%d %H:%M"
)

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

time_series = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cell

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(
    pd.api.types.is_float_dtype(time_series[c]) for c in COLUMNS.values()
), time_series.dtypes

# Snapshot taken before any other cell can touch the frame, so the closing self-check can prove
# nothing in between mutated it.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"Dataframe Shape     : {time_series.shape[0]:,} rows x {time_series.shape[1]} columns")
print(f"Dataframe Time Span : First({time_series.index.min()})  Last({time_series.index.max()})")
print(f"Monotonic index?    : {time_series.index.is_monotonic_increasing}")
print(f"Unique index?       : {time_series.index.is_unique}")

Dataframe Shape     : 67,336 rows x 11 columns
Dataframe Time Span : First(2019-01-01 00:00:00)  Last(2026-09-06 23:00:00)
Monotonic index?    : True
Unique index?       : True


In [3]:
# Our base features we get from the smard API
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_residual_load",
    "cap_wind_off", "cap_wind_on", "cap_solar"
]

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

# Basic engineered features
time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH),
    categories=SEASON_ORDER,
    ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)

# Outputs True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

# Basic engineered feature list
DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

# Check if more/less features were added/removed
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

11 data columns + 10 derived = 21 columns
YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


### 1.2 Resolution and durations

Every window and lag in this notebook is a **duration**, converted to an observation count from the
**measured** resolution by `n_obs`. A switch to SMARD's 15-min data changes the counts, not the
meaning. Repeated from the regression notebook §1.2.

In [4]:
# Calculate resolution of the smard time series
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

DAY_COMPLETENESS = 23 / 24   # accepts the spring-DST day, rejects materially short days


def n_obs(duration):
    """Observation count of `duration` at the measured resolution. Refuses a non-multiple."""
    count = duration / RESOLUTION
    if count != int(count):
        raise ValueError(f"{duration} is not a whole multiple of the resolution {RESOLUTION}")
    return int(count)


def describe(value):
    """Short readable form of a configuration value: hours below two days, whole days above."""
    if isinstance(value, pd.Timedelta):
        if value >= pd.Timedelta(days=2) and value == pd.Timedelta(days=value.days):
            return f"{value.days} days"
        return f"{value / pd.Timedelta(hours=1):g} h"
    if isinstance(value, pd.DateOffset):
        return ", ".join(f"{n} {unit.rstrip('s') if n == 1 else unit}" for unit, n in value.kwds.items())
    return str(value)


EXPECTED_OBS_PER_DAY = n_obs(pd.Timedelta(days=1))
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

print(f"Resolution =  {RESOLUTION}")
print(f"Data Resolution Observations : {describe(RESOLUTION)} = {EXPECTED_OBS_PER_DAY} observations per full day")
print(f"Data Day completeness        : >= {MIN_OBS_PER_DAY} of {EXPECTED_OBS_PER_DAY} observations (to accept each spring daylight saving time gap)")

Resolution =  0 days 01:00:00
Data Resolution Observations : 1 h = 24 observations per full day
Data Day completeness        : >= 23 of 24 observations (to accept each spring daylight saving time gap)


### 1.3 Configuration

- Every value later cells depend on is set **here and nowhere else**.

| Cell | Description |
|---|---|
| `DATA_INFO` | forecast setting: issue time on `DAY−1` and the publication lag of the actuals. **Must match `regression-models-claude.ipynb` §1.3.** The exported `cutoff` column lets a consumer check this. The 2 h lag is a **team trial**, not final (spec 06's default is 3 h): after a change, re-run this notebook. |
| `FEATURE_WINDOWS` | the same-hour lags, the two rolling windows and the annual period, as durations |
| `SEED`, `LEAKAGE_SAMPLE_DAYS` | the random draw of the leakage test (§4) |

The capacity publication rule of the regression notebook is not needed here: the capacity factor is
read at a past hour, together with the generation it divides (§3).

In [5]:
# Keep in sync with regression-models-claude.ipynb §1.3: this export is only valid for that setting.
DATA_INFO = {
    "issue_days_before": pd.Timedelta(days=1),     # the forecast for DAY is issued on DAY−1 ...
    "issue_clock": pd.Timedelta(hours=18),         # ... at 18:00 local, once both SMARD components are out
    "actuals_lag": pd.Timedelta(hours=2),          # team trial, not final (spec 06 default: 3 h); smard.de showed ~2 h
}

FEATURE_WINDOWS = {
    "same_hour_lags": [pd.Timedelta(days=2), pd.Timedelta(days=7)],   # DAY−2 replaces spec 05's DAY−1
    "roll_mean": pd.Timedelta(hours=72),
    "roll_std": pd.Timedelta(hours=168),
    "year_period": pd.Timedelta(days=365.25),
}

SEED = 42                  # every random draw in this notebook
LEAKAGE_SAMPLE_DAYS = 20   # delivery days rebuilt from truncated data in §4

cutoff_clock = DATA_INFO["issue_clock"] - DATA_INFO["actuals_lag"]
issue_day = f"DAY−{DATA_INFO['issue_days_before'].days}"

print("DATA_INFO")
print(f"  issue time             : {pd.Timestamp(0) + DATA_INFO['issue_clock']:%H:%M} on {issue_day}")
print(f"  actuals lag            : {describe(DATA_INFO['actuals_lag'])}")
print(f"  availability cutoff    : {pd.Timestamp(0) + cutoff_clock:%H:%M} on {issue_day}")
print(f"  last usable actual row : {pd.Timestamp(0) + cutoff_clock - RESOLUTION:%H:%M} on {issue_day}")
print("FEATURE_WINDOWS")
for name, value in FEATURE_WINDOWS.items():
    shown = ", ".join(describe(v) for v in value) if isinstance(value, list) else describe(value)
    print(f"  {name:<15}: {shown}")
print(f"leakage test: {LEAKAGE_SAMPLE_DAYS} days, seed {SEED}")

DATA_INFO
  issue time             : 18:00 on DAY−1
  actuals lag            : 2 h
  availability cutoff    : 16:00 on DAY−1
  last usable actual row : 15:00 on DAY−1
FEATURE_WINDOWS
  same_hour_lags : 2 days, 7 days
  roll_mean      : 3 days
  roll_std       : 7 days
  year_period    : 8766 h
leakage test: 20 days, seed 42


### 1.4 SMARD's hourly errors

`data/metrics/smard_forecast_errors_hourly.csv` is written by
[`forecast-metrics-claude.ipynb`](../02_forecast_metrics/forecast-metrics-claude.ipynb) (spec 04). It
is the source of one feature, the lagged wind + solar forecast error (`err_renewables`). It stays in
its own frame, `smard_errors`. The cell stops if the file is missing, or if its timestamps differ
from `time_series.index`, which would mean an export from another snapshot of `data/smard.csv`.

In [6]:
SMARD_ERRORS = DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv"
SMARD_ERRORS_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SMARD_ERROR_COLUMNS = ["err_renewables"]   # the only column this notebook uses

if not SMARD_ERRORS.exists():
    raise FileNotFoundError(
        f"{SMARD_ERRORS} not found. data/metrics/ is gitignored, so the file is not in a fresh clone — "
        f"regenerate it by running {SMARD_ERRORS_SOURCE} top to bottom."
    )

smard_errors = pd.read_csv(SMARD_ERRORS)
smard_errors["timestamp"] = pd.to_datetime(smard_errors["timestamp"], format="%Y-%m-%d %H:%M:%S")
smard_errors = smard_errors.set_index("timestamp")

missing_columns = sorted(set(SMARD_ERROR_COLUMNS) - set(smard_errors.columns))
if missing_columns:
    raise ValueError(f"{SMARD_ERRORS.name} lacks {missing_columns} — re-run {SMARD_ERRORS_SOURCE}.")

if not smard_errors.index.equals(time_series.index):
    raise ValueError(
        f"{SMARD_ERRORS.name} does not match data/smard.csv: "
        f"{len(smard_errors.index.difference(time_series.index)):,} timestamps only in the errors file, "
        f"{len(time_series.index.difference(smard_errors.index)):,} only in smard.csv "
        f"(errors file {smard_errors.index.min()} -> {smard_errors.index.max()}). It is a stale export "
        f"from another snapshot — re-run {SMARD_ERRORS_SOURCE} top to bottom."
    )

print(f"smard_errors    : {len(smard_errors):,} rows, index identical to time_series")
print(f"range           : {smard_errors.index.min()}  ->  {smard_errors.index.max()}")
print(f"columns used    : {SMARD_ERROR_COLUMNS}")

smard_errors    : 67,336 rows, index identical to time_series
range           : 2019-01-01 00:00:00  ->  2026-09-06 23:00:00
columns used    : ['err_renewables']


### 1.5 Initial self-check

Structural checks on the loaded data and the configuration, free of any hardcoded row count or date.

In [7]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())

assert set(DATA_INFO) == {"issue_days_before", "issue_clock", "actuals_lag"}, DATA_INFO
assert pd.Timedelta(0) < DATA_INFO["actuals_lag"] < DATA_INFO["issue_clock"], "the cutoff must fall on the issue day"
assert all(isinstance(v, pd.Timedelta) for v in [*FEATURE_WINDOWS["same_hour_lags"], FEATURE_WINDOWS["roll_mean"],
                                                   FEATURE_WINDOWS["roll_std"], FEATURE_WINDOWS["year_period"]])
for window in ["roll_mean", "roll_std"]:
    n_obs(FEATURE_WINDOWS[window])   # refuses a window that is not a whole number of observations

print("setup self-check passed")
print(f"  {len(SERIES)} float series, index sorted and unique, columns == SERIES + DERIVED")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

setup self-check passed
  11 float series, index sorted and unique, columns == SERIES + DERIVED
  67,336 rows, 2019-01-01 00:00:00 -> 2026-09-06 23:00:00, years [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


---

## 2 Forecast setting

Repeated from the regression notebook §2, so every feature here obeys the same rule for **what is
known when**. Values from `DATA_INFO` in brackets:

| Item | Rule |
|---|---|
| `ISSUE_TIME` | the forecast for delivery day `DAY` is issued on `DAY−1` (18:00 local) |
| `AVAILABILITY_CUTOFF` | `ISSUE_TIME − actuals lag` (currently a 2 h trial, so 16:00 on `DAY−1`; spec 06's default of 3 h gives 15:00) |
| actuals and SMARD's errors | a row stamped `t` (interval start) is usable if `t + resolution ≤ AVAILABILITY_CUTOFF`: up to the row stamped 15:00 |
| `Forecast Wind + Solar` for `DAY` | available: published by 18:00 on `DAY−1` |
| calendar facts about `DAY` | always available |

Spec 05's origin was 00:00 on `DAY`, i.e. actuals up to 23:00 on `DAY−1`. With the current 2 h lag,
the eight hours from 16:00 to 23:00 on `DAY−1` are the difference, and every spec 05 look-back that touches them has to
move.

### 2.1 The forecast-setting helper

`forecast_setting(days)` returns, per delivery day, the issue time, the availability cutoff and the
target hours. `SETTING` holds it for every day of the record, `ROW_SETTING` the same times per row.

In [8]:
# Delivery day of every row: its local calendar date as a local midnight (the DERIVED "date" boundary).
DAY_OF_ROW = time_series.index.normalize()
TARGET_HOURS = time_series.index.groupby(DAY_OF_ROW)   # delivery day -> its local hours in the record
DAYS = pd.DatetimeIndex(sorted(TARGET_HOURS))


def forecast_setting(days):
    """The forecast setting of each delivery day in `days` (local midnights).

    One row per day: `issue_time` (ISSUE_TIME), `cutoff` (AVAILABILITY_CUTOFF) and `target_hours`,
    the day's local hours as they exist in the record. Every model, feature and training row takes
    its times from here.
    """
    days = pd.DatetimeIndex(days)
    issue_time = days - DATA_INFO["issue_days_before"] + DATA_INFO["issue_clock"]
    return pd.DataFrame(
        {
            "issue_time": issue_time,
            "cutoff": issue_time - DATA_INFO["actuals_lag"],
            "target_hours": [TARGET_HOURS[day] for day in days],
        },
        index=days,
    )


def available(stamps, cutoff):
    """True where an observation stamped `stamps` (interval start) has ended by `cutoff`."""
    return stamps + RESOLUTION <= cutoff


SETTING = forecast_setting(DAYS)
ROW_SETTING = pd.DataFrame(
    {
        "day": DAY_OF_ROW,
        "issue_time": SETTING["issue_time"].reindex(DAY_OF_ROW).to_numpy(),
        "cutoff": SETTING["cutoff"].reindex(DAY_OF_ROW).to_numpy(),
    },
    index=time_series.index,
)

hours_per_day = SETTING["target_hours"].map(len)
dst_days = SETTING.index[hours_per_day < EXPECTED_OBS_PER_DAY]


def setting_summary(day):
    """One readable row of the forecast setting for delivery day `day`."""
    row = SETTING.loc[day]
    last_actual = row["cutoff"] - RESOLUTION
    # Counted in rows, not clock hours: the missing spring-DST hour shortens the horizon by one row.
    last_position = time_series.index.searchsorted(last_actual, side="right") - 1
    rows_ahead = time_series.index.get_indexer(row["target_hours"]) - last_position
    return {
        "DAY": f"{day:%Y-%m-%d} ({DAY_NAMES[day.dayofweek]})",
        "ISSUE_TIME": f"{row['issue_time']:%Y-%m-%d %H:%M}",
        "AVAILABILITY_CUTOFF": f"{row['cutoff']:%Y-%m-%d %H:%M}",
        "last usable actual row": f"{last_actual:%Y-%m-%d %H:%M}",
        "target hours": len(row["target_hours"]),
        "rows after last usable actual": f"{rows_ahead.min()}–{rows_ahead.max()}",
    }


print(f"delivery days     : {len(SETTING):,}  ({DAYS[0]:%Y-%m-%d} .. {DAYS[-1]:%Y-%m-%d})")
print(f"target hours/day  : {hours_per_day.value_counts().sort_index(ascending=False).to_dict()}")
print(f"spring DST days   : {len(dst_days)}, each with {hours_per_day[dst_days].min()} target hours")
print("\nForecast setting of two delivery days: the record's latest spring DST day and its last day")
display(pd.DataFrame([setting_summary(dst_days[-1]), setting_summary(DAYS[-1])]).set_index("DAY"))

delivery days     : 2,806  (2019-01-01 .. 2026-09-06)
target hours/day  : {24: 2798, 23: 8}
spring DST days   : 8, each with 23 target hours

Forecast setting of two delivery days: the record's latest spring DST day and its last day


,ISSUE_TIME,AVAILABILITY_CUTOFF,last usable actual row,target hours,rows after last usable actual
DAY,,,,,
2026-03-29 (Sun),2026-03-28 18:00,2026-03-28 16:00,2026-03-28 15:00,23,9–31
2026-09-06 (Sun),2026-09-05 18:00,2026-09-05 16:00,2026-09-05 15:00,24,9–32


### 2.2 Look-backs against the availability rule

Every same-hour lag in `FEATURE_WINDOWS` is checked on **every row** of the record. A lag that
reaches past the cutoff stops the notebook instead of leaking silently. Spec 05's 24 h lag is shown
next to them, **not** as a feature, only to show what the rule rejects. The two rolling windows
end at the last usable row by construction; the leakage test (§4) proves the built features do.

In [9]:
cutoffs = ROW_SETTING["cutoff"].to_numpy()


def lag_availability(lag):
    """How a same-hour lag of `lag` fares against the cutoff, over every row of the record."""
    source = time_series.index - lag
    usable = available(source, cutoffs)
    return usable, {
        "rows past the cutoff": f"{(~usable).sum():,} of {len(usable):,}",
        "smallest margin to the cutoff": describe((cutoffs - (source + RESOLUTION)).min()),
    }


availability = {}
for lag in FEATURE_WINDOWS["same_hour_lags"]:
    usable, row = lag_availability(lag)
    if not usable.all():
        raise ValueError(
            f"same-hour lag {describe(lag)} reaches past the availability cutoff for {(~usable).sum():,} rows: "
            "it would leak. Use a lag the forecast setting allows."
        )
    availability[f"same local hour, {describe(lag)} earlier"] = {"status": "feature", **row}

# Spec 05's lag, for comparison only: never built as a feature here.
spec05_lag = pd.Timedelta(hours=24)
usable_24h, row = lag_availability(spec05_lag)
leak_hours = sorted(set(time_series.index[~usable_24h].hour))
availability[f"same local hour, {describe(spec05_lag)} earlier (spec 05)"] = {"status": "rejected", **row}

last_row = f"{pd.Timestamp(0) + cutoff_clock - RESOLUTION:%H:%M} on {issue_day}"
for name in ["roll_mean", "roll_std"]:
    availability[f"trailing {describe(FEATURE_WINDOWS[name])} window"] = {
        "status": "feature",
        "rows past the cutoff": f"none: ends at the row stamped {last_row}",
        "smallest margin to the cutoff": describe(pd.Timedelta(0)),
    }

print("Look-backs against the availability rule (lags checked on every row of the record)")
display(pd.DataFrame(availability).T.rename_axis("look-back"))
print(f"target hours at which the 24 h lag leaks: {leak_hours[0]:02d}:00 .. {leak_hours[-1]:02d}:00")

Look-backs against the availability rule (lags checked on every row of the record)


,status,rows past the cutoff,smallest margin to the cutoff
look-back,,,
"same local hour, 2 days earlier",feature,"0 of 67,336",16 h
"same local hour, 7 days earlier",feature,"0 of 67,336",136 h
"same local hour, 24 h earlier (spec 05)",rejected,"22,448 of 67,336",-8 h
trailing 3 days window,feature,none: ends at the row stamped 15:00 on DAY−1,0 h
trailing 7 days window,feature,none: ends at the row stamped 15:00 on DAY−1,0 h


target hours at which the 24 h lag leaks: 16:00 .. 23:00


**Reading the table.** The numbers are for the current 2 h trial lag; a different `actuals_lag`
moves the cutoff and changes them.

- **Spec 05's 24 h lag leaks in exactly one third of all hours.**
  - It reaches past the cutoff in 22,448 of 67,336 rows: the eight target hours 16:00–23:00 on every
    one of the 2,806 delivery days (8 × 2,806 = 22,448).
  - Its worst row has a margin of −8 h: target hour 23:00 reads the row stamped 23:00 on `DAY−1`,
    whose interval ends at midnight, 8 h after the 16:00 cutoff.

- **The two-day lag that replaces it is safe on every row, with room to spare.**
  - Its smallest margin is 16 h. The latest source it reads is 23:00 on `DAY−2`, and that
    interval ends at 00:00 on `DAY−1`, 16 h before the cutoff.
  - The margin would only run out if the actuals lag exceeded 18 h, so a change to
    `actuals_lag` in `DATA_INFO` does not threaten it.

- **The rolling windows sit right at the cutoff (margin 0 h).**
  - Both end at the row stamped 15:00 on `DAY−1`, the last usable row. There is no margin by design,
    which is why §4 re-tests them on truncated data rather than trusting the construction.

### 2.3 Self-check

In [10]:
assert SETTING.index.equals(DAYS) and DAYS.is_unique and DAYS.is_monotonic_increasing
assert (SETTING["issue_time"] - SETTING["cutoff"] == DATA_INFO["actuals_lag"]).all()
assert (SETTING["issue_time"] < SETTING.index).all(), "a forecast is issued after its delivery day starts"
assert sum(map(len, SETTING["target_hours"])) == len(time_series), "target hours do not partition the record"
assert all((hours.normalize() == day).all() for day, hours in SETTING["target_hours"].items())
assert ROW_SETTING.index.equals(time_series.index) and ROW_SETTING.notna().all().all()
assert not usable_24h.all(), "spec 05's 24 h lag should leak under this cutoff; check DATA_INFO"
assert list(time_series.columns) == SERIES + DERIVED, "a section 2 cell persisted a column onto time_series"

print("section 2 self-check passed")
print(f"  {len(SETTING):,} delivery days, target hours partition the record, cutoff = issue time − {describe(DATA_INFO['actuals_lag'])}")
print(f"  every same-hour lag in FEATURE_WINDOWS is available on every row; spec 05's 24 h lag is not")

section 2 self-check passed
  2,806 delivery days, target hours partition the record, cutoff = issue time − 2 h
  every same-hour lag in FEATURE_WINDOWS is available on every row; spec 05's 24 h lag is not


---

## 3 Features

One function, `build_features(frame, errors)`, builds all 16 columns for every row of `frame`. It
reads **only its two arguments**, never the globals `time_series` or `smard_errors`. That is what
lets §4 call it on data cut at a day's cutoff: a feature that peeks past the cutoff then changes.

### 3.1 Mapping to spec 05

| Spec 05 feature | Here | Change and reason |
|---|---|---|
| `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `doy_sin`, `doy_cos` | same names | none: calendar facts about `DAY` |
| `is_weekend`, `is_holiday` | same names | none |
| `season` | `season_code` | the position in `SEASON_ORDER` (0 = winter … 3 = autumn), a number for models that need one |
| `residual_load_lag_24h` | `residual_load_lag_48h` | the 24 h lag leaks for target hours from the cutoff clock on: 16:00–23:00 with the current 2 h lag (§2.2). `DAY−2` is the nearest same-hour lag that never does. |
| `residual_load_lag_168h` | same name | none: safe with a 136 h margin (§2.2) |
| `vre_capacity_factor_lag_24h` | `vre_capacity_factor_lag_48h` | same leak, same move. Capacity is read at the past hour, together with the generation it divides, as in spec 05. |
| `forecast_wind_solar_error_lag_24h` | `forecast_wind_solar_error_lag_48h` | same leak, same move. The error needs the actual, so it follows the actuals rule. |
| `residual_load_roll_mean_72h` | `residual_load_roll_mean_72h_at_cutoff` | spec 05's window ends at 23:00 on `DAY−1`. Here it ends at the last usable row (15:00 on `DAY−1` with the current 2 h lag). |
| `residual_load_roll_std_168h` | `residual_load_roll_std_168h_at_cutoff` | same |
| `forecast_wind_solar` | same name | none: `fc_gen_wind_solar` for the target hour is published by 18:00 on `DAY−1` |

The `_at_cutoff` suffix keeps the rolling columns apart from spec 05's same-named columns in
`data/features/residual_load_features.csv`, which end at a different hour.

Spec 05's two **dropped** candidates (`wind_solar_lag_24h`, `wind_solar_share`) and its three
**excluded** inputs (`fc_grid_load`, `fc_residual_load`, the target day's actuals) stay dropped and
excluded, for the reasons given in
[`05-feature-engineering.md`](../../.claude/specs/05-feature-engineering.md).

### 3.2 Construction rules

- **Calendar and cyclical**: from each row's own local timestamp. The time of day is the elapsed
  share of the local day, which equals `hour / 24` at hourly resolution.
- **Same-hour lags**: a wall-clock shift, the value at the same local time `lag` earlier. A source
  hour that does not exist (local 02:00 on a spring-DST day) stays empty.
- **Rolling**: one value per delivery day, from the trailing window ending at that day's last usable
  row, broadcast to all its hours. `min_periods` is the full observation count of the window, so a
  window that spans a spring-DST gap is one row short and stays empty, as in spec 05.
- Nothing is filled or interpolated.

In [11]:
DE_HOLIDAYS = holidays.country_holidays("DE", years=YEARS)   # one source of truth, no subdiv
CAP_COLUMNS = ["cap_wind_off", "cap_wind_on", "cap_solar"]
MEASURED_COLUMNS = ["wind_off", "wind_on", "solar", "grid_load", "residual_load", "renewables", *CAP_COLUMNS]

LAG_2D, LAG_7D = FEATURE_WINDOWS["same_hour_lags"]
FEATURE_COLUMNS = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos",
    "is_weekend", "is_holiday", "season_code",
    "residual_load_lag_48h", "residual_load_lag_168h",
    "vre_capacity_factor_lag_48h", "forecast_wind_solar_error_lag_48h",
    "residual_load_roll_mean_72h_at_cutoff", "residual_load_roll_std_168h_at_cutoff",
    "forecast_wind_solar",
]


def row_cutoff(index):
    """AVAILABILITY_CUTOFF of each row's delivery day, from DATA_INFO (the same rule as forecast_setting)."""
    day = index.normalize()
    return day - DATA_INFO["issue_days_before"] + DATA_INFO["issue_clock"] - DATA_INFO["actuals_lag"]


def same_hour_lag(series, index, lag):
    """`series` at the same local wall-clock time `lag` earlier, aligned onto `index`. Missing source -> NaN."""
    return series.reindex(index - lag).to_numpy()


def at_last_usable_row(series, index, window, stat):
    """`stat` of `series` over the trailing `window` that ends at each row's last usable row (cutoff − resolution)."""
    rolled = getattr(series.rolling(window, min_periods=n_obs(window)), stat)()
    return rolled.reindex(row_cutoff(index) - RESOLUTION).to_numpy()


def build_features(frame, errors):
    """All 16 feature columns for every row of `frame`. Reads only `frame` and `errors`."""
    index = frame.index
    day = index.normalize()
    time_of_day = (index - day) / pd.Timedelta(days=1)          # 0 .. <1, = hour / 24 at hourly resolution
    year_share = index.dayofyear / (FEATURE_WINDOWS["year_period"] / pd.Timedelta(days=1))
    capacity_factor = frame["renewables"] / frame[CAP_COLUMNS].sum(axis=1, min_count=len(CAP_COLUMNS))

    out = pd.DataFrame(index=index)
    out["hour_sin"], out["hour_cos"] = np.sin(2 * np.pi * time_of_day), np.cos(2 * np.pi * time_of_day)
    out["dow_sin"], out["dow_cos"] = np.sin(2 * np.pi * index.dayofweek / 7), np.cos(2 * np.pi * index.dayofweek / 7)
    out["doy_sin"], out["doy_cos"] = np.sin(2 * np.pi * year_share), np.cos(2 * np.pi * year_share)
    out["is_weekend"] = (index.dayofweek >= 5).astype(int)
    out["is_holiday"] = pd.Index(day.date).isin(set(DE_HOLIDAYS)).astype(int)
    out["season_code"] = index.month.map(SEASON_OF_MONTH).map(SEASON_ORDER.index).astype(int)
    out["residual_load_lag_48h"] = same_hour_lag(frame["residual_load"], index, LAG_2D)
    out["residual_load_lag_168h"] = same_hour_lag(frame["residual_load"], index, LAG_7D)
    out["vre_capacity_factor_lag_48h"] = same_hour_lag(capacity_factor, index, LAG_2D)
    out["forecast_wind_solar_error_lag_48h"] = same_hour_lag(errors["err_renewables"], index, LAG_2D)
    out["residual_load_roll_mean_72h_at_cutoff"] = at_last_usable_row(
        frame["residual_load"], index, FEATURE_WINDOWS["roll_mean"], "mean")
    out["residual_load_roll_std_168h_at_cutoff"] = at_last_usable_row(
        frame["residual_load"], index, FEATURE_WINDOWS["roll_std"], "std")
    out["forecast_wind_solar"] = frame["fc_gen_wind_solar"].to_numpy()
    return out[FEATURE_COLUMNS]


assert row_cutoff(time_series.index).equals(pd.DatetimeIndex(ROW_SETTING["cutoff"])), "row_cutoff disagrees with §2.1"

features = build_features(time_series, smard_errors)

print(f"features: {features.shape[0]:,} rows x {features.shape[1]} columns, on time_series.index")
last_day = SETTING.index[-1]
example_hours = SETTING.at[last_day, "target_hours"][[0, 15, 16, 23]]
print(f"\nFour target hours of the record's last delivery day ({last_day:%Y-%m-%d}); "
      f"15:00 and 16:00 straddle the cutoff clock (MWh; ratios and encodings dimensionless)")
display(features.loc[example_hours].T.rename(columns=lambda t: f"{t:%H:%M}"))

features: 67,336 rows x 16 columns, on time_series.index

Four target hours of the record's last delivery day (2026-09-06); 15:00 and 16:00 straddle the cutoff clock (MWh; ratios and encodings dimensionless)


timestamp,00:00,15:00,16:00,23:00
hour_sin,0.000000,-0.707107,-0.866025,-0.258819
hour_cos,1.000000,-0.707107,-0.500000,0.965926
dow_sin,-0.781831,-0.781831,-0.781831,-0.781831
dow_cos,0.623490,0.623490,0.623490,0.623490
doy_sin,-0.909388,-0.909388,-0.909388,-0.909388
doy_cos,-0.415948,-0.415948,-0.415948,-0.415948
is_weekend,1.000000,1.000000,1.000000,1.000000
is_holiday,0.000000,0.000000,0.000000,0.000000
season_code,3.000000,3.000000,3.000000,3.000000
residual_load_lag_48h,16548.930000,-3059.070000,-2950.260000,8804.470000


### 3.3 Missing values

Every empty value has a reason from the data: the start of the record, a spring-DST gap, or a missing
SMARD value. The table counts them per column, in hours and in delivery days.

In [12]:
missing_hours = features.isna().sum()
missing_days = features.isna().groupby(features.index.normalize()).any().sum()
missing = pd.DataFrame({"empty hours": missing_hours, "empty days (any hour)": missing_days})
missing = missing[missing["empty hours"] > 0]

first_empty = {c: features.index[features[c].isna()][0] for c in missing.index}
last_empty = {c: features.index[features[c].isna()][-1] for c in missing.index}
missing["first empty"] = pd.Series(first_empty).dt.strftime("%Y-%m-%d %H:%M")
missing["last empty"] = pd.Series(last_empty).dt.strftime("%Y-%m-%d %H:%M")

print(f"Empty values per feature over the record ({len(features):,} hours, {len(SETTING):,} delivery days, "
      f"{len(dst_days)} spring-DST days); columns without gaps are not listed")
display(missing)

Empty values per feature over the record (67,336 hours, 2,806 delivery days, 8 spring-DST days); columns without gaps are not listed


,empty hours,empty days (any hour),first empty,last empty
residual_load_lag_48h,56,10,2019-01-01 00:00,2026-03-31 02:00
residual_load_lag_168h,176,15,2019-01-01 00:00,2026-04-05 02:00
vre_capacity_factor_lag_48h,56,10,2019-01-01 00:00,2026-03-31 02:00
forecast_wind_solar_error_lag_48h,56,10,2019-01-01 00:00,2026-03-31 02:00
residual_load_roll_mean_72h_at_cutoff,672,28,2019-01-01 00:00,2026-04-01 23:00
residual_load_roll_std_168h_at_cutoff,1536,64,2019-01-01 00:00,2026-04-05 23:00


**Reading the table.** Computed with the current 2 h trial lag. A different lag can shift at most
the warm-up counts of the rolling features. Every gap is explained by the start of the record or by the 8 spring-DST
days. None comes from a missing SMARD value, and the calendar columns and `forecast_wind_solar`
have no gaps at all.

- **The same-hour lags lose their warm-up plus one hour per DST day.**
  - `residual_load_lag_48h`: 56 empty hours = the record's first 48 h + 8 hours whose source is a
    non-existent local 02:00 (last one 2026-03-31 02:00, two days after the latest DST switch).
  - `residual_load_lag_168h`: 176 = 168 + 8, in the same pattern.
  - `vre_capacity_factor_lag_48h` and `forecast_wind_solar_error_lag_48h` match the 48 h lag
    exactly (56 hours), so `err_renewables` has no gap of its own in this snapshot.

- **The rolling features lose whole days: warm-up plus every window that spans a DST gap.**
  - `…_roll_mean_72h_at_cutoff`: 28 days (672 h) = 4 warm-up days + 3 days after each of the 8 DST
    switches.
  - `…_roll_std_168h_at_cutoff`: 64 days (1,536 h) = 8 warm-up days + 7 days after each DST switch.
  - That is one warm-up day more than spec 05 reports (27 and 63 days). The window now ends 8 h
    earlier, at 15:00 on `DAY−1`, so one more early day's window reaches back before the first row.

- **2,742 of 2,806 delivery days (97.7 %) have every feature defined.** The 64 others are exactly
  the days the 168 h window leaves empty; every other gap falls inside them.

---

## 4 Leakage test by truncation

The same test as the regression notebook §5.4 (spec 06 Behaviour 8). For each sampled delivery day,
the data is **cut off at that day's availability cutoff**:

- measured columns (generation, load, residual load, `renewables`, capacity) and SMARD's errors are
  emptied from the cutoff on
- `fc_gen_wind_solar` is kept up to the end of `DAY` (it is published by 18:00 on `DAY−1`)
- every row after `DAY` is dropped

The day's features are rebuilt from the cut data and must be **identical** to the features built
from the full data. A feature that reads anything after the cutoff changes when that data is gone,
so this catches leakage without tracking a timestamp per feature.

The sample holds `LEAKAGE_SAMPLE_DAYS` days drawn with `SEED` from the days whose features are all
defined, plus the two days where leakage is easiest to get wrong: the record's latest 1 January and
latest spring-DST day.

**Control.** The same test is run once on spec 05's 24 h lag, which §2.2 showed reaches past the
cutoff. It must **fail**. A test that cannot fail would prove nothing.

In [13]:
def cut_at_cutoff(day):
    """The data as known at `day`'s issue time: measured values and errors up to the cutoff, forecasts to the end of `day`."""
    cutoff, end = SETTING.at[day, "cutoff"], SETTING.at[day, "target_hours"][-1]
    frame, errors = time_series.loc[:end].copy(), smard_errors.loc[:end].copy()
    frame.loc[~available(frame.index, cutoff), MEASURED_COLUMNS] = np.nan
    errors.loc[~available(errors.index, cutoff), :] = np.nan   # every error needs the actual
    return frame, errors


complete_days = SETTING.index[features.notna().all(axis=1).groupby(features.index.normalize()).all().to_numpy()]
rng = np.random.default_rng(SEED)
SAMPLE = pd.DatetimeIndex(sorted(rng.choice(complete_days, size=LEAKAGE_SAMPLE_DAYS, replace=False)))
new_year = DAYS[(DAYS.month == 1) & (DAYS.day == 1)][-1]
EDGE_DAYS = {"latest 1 January": new_year, "latest spring-DST day": dst_days[-1]}
SAMPLE = SAMPLE.union(pd.DatetimeIndex(list(EDGE_DAYS.values())))

checks = []
for day in SAMPLE:
    frame, errors = cut_at_cutoff(day)
    hours = SETTING.at[day, "target_hours"]
    rebuilt = build_features(frame, errors).loc[hours]
    changed = [c for c in FEATURE_COLUMNS if not rebuilt[c].equals(features.loc[hours, c])]
    checks.append({
        "day": f"{day:%Y-%m-%d}",
        "why sampled": next((k for k, v in EDGE_DAYS.items() if v == day), "random"),
        "target hours": len(hours),
        "empty feature values": int(features.loc[hours].isna().sum().sum()),
        "identical": not changed,
        "changed columns": ", ".join(changed) or "—",
    })
checks = pd.DataFrame(checks).set_index("day")

# Control: spec 05's 24 h lag, built the same way, on the first sampled day.
control_day = SAMPLE[0]
control_hours = SETTING.at[control_day, "target_hours"]
control_frame, _ = cut_at_cutoff(control_day)
spec05_full = time_series["residual_load"].reindex(control_hours - spec05_lag).to_numpy()
spec05_cut = control_frame["residual_load"].reindex(control_hours - spec05_lag).to_numpy()
control_changed = ~np.isclose(spec05_full, spec05_cut, equal_nan=True)

print(f"Leakage test: {len(SAMPLE)} delivery days rebuilt from data cut at their cutoff "
      f"({LEAKAGE_SAMPLE_DAYS} drawn with seed {SEED} from {len(complete_days):,} complete days, plus {len(EDGE_DAYS)} edge days)")
display(checks)
print(f"\nControl, spec 05's residual_load_lag_24h on {control_day:%Y-%m-%d}: "
      f"{control_changed.sum()} of {len(control_hours)} target hours change when the data after the cutoff is removed "
      f"({', '.join(f'{h:%H:%M}' for h in control_hours[control_changed])})")
print(f"test passes on every sampled day: {checks['identical'].all()}  ·  control fails as it must: {control_changed.any()}")

Leakage test: 22 delivery days rebuilt from data cut at their cutoff (20 drawn with seed 42 from 2,742 complete days, plus 2 edge days)


,why sampled,target hours,empty feature values,identical,changed columns
day,,,,,
2019-09-07,random,24,0,True,—
2019-09-16,random,24,0,True,—
2019-09-30,random,24,0,True,—
2020-01-02,random,24,0,True,—
2020-07-26,random,24,0,True,—
2022-05-01,random,24,0,True,—
2022-05-17,random,24,0,True,—
2022-06-24,random,24,0,True,—
2022-12-12,random,24,0,True,—



Control, spec 05's residual_load_lag_24h on 2019-09-07: 8 of 24 target hours change when the data after the cutoff is removed (16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00)
test passes on every sampled day: True  ·  control fails as it must: True


**Reading the test.** Computed with the current 2 h trial lag. The sampled days are the same for any
lag, but the hours at which the control changes follow the cutoff.

- **No feature reads anything after its cutoff.**
  - All 22 sampled days rebuild identically from the cut data. They span the record, from
    2019-09-07 to 2026-06-24, and include the latest 1 January (2026-01-01) and the latest
    spring-DST day (2026-03-29, 23 target hours).
  - This covers the two rolling features, whose windows end exactly at the last usable row (0 h
    margin in §2.2): the test would catch them if they reached even one row further.

- **The test does catch a real leak.**
  - On the control day 2019-09-07, spec 05's `residual_load_lag_24h` changes in 8 of 24 target
    hours, exactly 16:00–23:00, the hours §2.2 flagged. Under spec 06's forecast setting that
    feature would have used data that did not exist yet at 18:00 on `DAY−1`.

---

## 5 Export and self-check

### 5.1 Export

`data/features/residual_load_features_cutoff.csv`: one row per hour of `time_series`, with
`timestamp`, the row's `cutoff` and the 16 features. Plain CSV (`sep=","`, `decimal="."`, UTF-8),
the project's convention for derived files; a bare `pd.read_csv` reads it. `data/features/` is
gitignored (`data/features/*.csv`) and kept in git by an empty `.gitkeep`.

The `cutoff` column records the forecast setting the file was built for. A consumer, e.g. the
regression notebook, checks it against its own `ROW_SETTING["cutoff"]`: if the two `DATA_INFO` cells
drift apart, that check fails instead of the model silently training on a leaking feature.

In [14]:
FEATURES_DIR = DATA_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = FEATURES_DIR / "residual_load_features_cutoff.csv"

export = features.copy()
export.insert(0, "cutoff", ROW_SETTING["cutoff"].to_numpy())
export.rename_axis("timestamp").to_csv(OUTPUT_PATH, sep=",", decimal=".", encoding="utf-8")

print(f"wrote {OUTPUT_PATH.relative_to(DATA_DIR.parent)}: {len(export):,} rows x {export.shape[1] + 1} columns "
      f"(timestamp, cutoff, {len(FEATURE_COLUMNS)} features)")

wrote data/features/residual_load_features_cutoff.csv: 67,336 rows x 18 columns (timestamp, cutoff, 16 features)


### 5.2 Verify the export and closing self-check

In [15]:
reloaded = pd.read_csv(OUTPUT_PATH)
assert list(reloaded.columns) == ["timestamp", "cutoff", *FEATURE_COLUMNS], list(reloaded.columns)
reloaded["timestamp"] = pd.to_datetime(reloaded["timestamp"], format="%Y-%m-%d %H:%M:%S")
reloaded["cutoff"] = pd.to_datetime(reloaded["cutoff"], format="%Y-%m-%d %H:%M:%S")
reloaded = reloaded.set_index("timestamp")

# The file as written
assert reloaded.index.equals(time_series.index), "the export does not hold one row per time_series row"
assert (reloaded["cutoff"].to_numpy() == ROW_SETTING["cutoff"].to_numpy()).all(), "cutoff column differs from §2.1"
# A bare read_csv parses floats with pandas' fast parser, which can differ from the written value in the
# last binary digit (~1e-16). Anything beyond that relative tolerance would be a real difference.
pd.testing.assert_frame_equal(reloaded[FEATURE_COLUMNS], features, rtol=1e-12, atol=0, check_freq=False)

# The features
assert features.index.equals(time_series.index) and list(features.columns) == FEATURE_COLUMNS
cyclical = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos"]
assert features[cyclical].abs().le(1).all().all(), "a cyclical encoding lies outside [-1, 1]"
assert set(features["season_code"].unique()) <= set(range(len(SEASON_ORDER)))
assert set(features["is_weekend"].unique()) <= {0, 1} and set(features["is_holiday"].unique()) <= {0, 1}
assert features["vre_capacity_factor_lag_48h"].dropna().between(0, 1).all(), "a capacity factor outside [0, 1]"
assert checks["identical"].all(), "a feature changed when the data after its cutoff was removed: it leaks"
assert control_changed.any(), "the control did not fail: the leakage test cannot detect a leak"

# The inputs: no grid-load or residual-load forecast feeds any column
probe = time_series.copy()
probe[["fc_grid_load", "fc_residual_load"]] = np.nan
assert build_features(probe, smard_errors).equals(features), "fc_grid_load or fc_residual_load feeds a feature"

# Nothing was persisted onto time_series, and it is the frame that was loaded
assert list(time_series.columns) == SERIES + DERIVED, "a cell persisted a column onto time_series"
assert (len(time_series), time_series.index.min(), time_series.index.max()) == (LOADED["rows"], LOADED["start"], LOADED["end"])

print("closing self-check passed")
print(f"  {OUTPUT_PATH.name}: {len(reloaded):,} rows, identical to `features`, cutoff column matches the forecast setting")
print(f"  encodings in range, capacity factor in [0, 1], leakage test passed on {len(checks)} days, control failed")
print(f"  no feature depends on fc_grid_load or fc_residual_load; time_series still holds SERIES + DERIVED")

closing self-check passed
  residual_load_features_cutoff.csv: 67,336 rows, identical to `features`, cutoff column matches the forecast setting
  encodings in range, capacity factor in [0, 1], leakage test passed on 22 days, control failed
  no feature depends on fc_grid_load or fc_residual_load; time_series still holds SERIES + DERIVED
